# **Semantic search**

In [1]:
import chromadb

In [2]:
from pathlib import Path
path = Path(".").resolve().parent / "Collections"
print(path)

C:\Users\aliai\Desktop\M2 I2A\RAG Agentique\Backend\Collections


In [3]:
client = chromadb.PersistentClient(path=path)
collection = client.get_collection(
    name="jurisprudence"
)

## **Getting the docs from the collection**

In [4]:
stored = collection.get(include=["documents", "metadatas"])
keyword_documents = stored["documents"]
if not keyword_documents:
    raise ValueError("The Chroma collection contains no documents")
if not (len(keyword_documents) == len(stored["ids"]) == len(stored["metadatas"])):
    raise ValueError("Chroma returned misaligned documents, IDs, and metadata")

keyword_index = {
    "documents": keyword_documents,
    "ids": stored["ids"],
    "metadatas": stored["metadatas"],
}

## **Helping functions**

In [7]:
def _rank_cases(scores, candidate_indices, index, top_k):
    if top_k <= 0:
        raise ValueError("top_k must be a positive integer")

    best_chunks = {}
    for chunk_index in candidate_indices:
        metadata = index["metadatas"][chunk_index]
        source_id = metadata.get("source_id")
        if not source_id:
            raise ValueError(f"Chunk {index['ids'][chunk_index]} has no source_id metadata")

        score = float(scores[chunk_index])
        current = best_chunks.get(source_id)
        if current is None or score > current["score"]:
            best_chunks[source_id] = {
                "document": index["documents"][chunk_index],
                "score": score,
                "id": index["ids"][chunk_index],
                "source_id": source_id,
                "metadatas": metadata,
            }

    ranked_cases = sorted(
        best_chunks.values(),
        key=lambda result: (result["score"], result["source_id"]),
    )
    return ranked_cases[:top_k]

In [ ]:
def semanticSearch( index: dict, query : str, n_results : int = 50, top_k : int = 5):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
    )
    
    scores = results["distances"][0]
    ids = results["ids"][0]
    candidate_indices = []
    scores_dict = [0]*len(index["ids"])
    for id in ids:
        i = index["ids"].index(id)
        candidate_indices.append(i)
        scores_dict[i] = scores[ids.index(id)]
    ranked_cases = _rank_cases(scores_dict, candidate_indices, index, top_k)
    return ranked_cases

In [ ]:
from math import log2
from statistics import mean

K_VALUES = (5, 10, 20, 30)


def evaluate_strategy(search_function, method_name, evaluation_queries, index):
    per_query = []
    max_k = max(K_VALUES)

    for query_data in evaluation_queries:
        relevant_ids = set(query_data["relevant_source_ids"])
        if not relevant_ids:
            raise ValueError(f"{query_data['query_id']} has no relevance labels")

        ranked_results = search_function(
            query=query_data["query"],
            top_k=max_k,
            index=index,
        )
        ranked_ids = [result["source_id"] for result in ranked_results]
        if len(ranked_ids) != len(set(ranked_ids)):
            raise ValueError(f"{method_name} returned duplicate cases")

        row = {
            "method": method_name,
            "query_id": query_data["query_id"],
            "query": query_data["query"],
            "retrieved_source_ids": ranked_ids,
        }
        relevant_ranks = [
            rank for rank, source_id in enumerate(ranked_ids, start=1)
            if source_id in relevant_ids
        ]
        row["mrr@10"] = 1 / relevant_ranks[0] if relevant_ranks else 0.0

        for k in K_VALUES:
            found = sum(source_id in relevant_ids for source_id in ranked_ids[:k])
            row[f"precision@{k}"] = found / k
            row[f"recall@{k}"] = found / len(relevant_ids)

            dcg = sum(
                1 / log2(rank + 1)
                for rank, source_id in enumerate(ranked_ids[:k], start=1)
                if source_id in relevant_ids
            )
            ideal_count = min(len(relevant_ids), k)
            ideal_dcg = sum(
                1 / log2(rank + 1)
                for rank in range(1, ideal_count + 1)
            )
            row[f"ndcg@{k}"] = dcg / ideal_dcg if ideal_dcg else 0.0

        per_query.append(row)

    summary = {"method": method_name, "queries": len(per_query)}
    for metric in (
        "precision@5", "recall@5", "ndcg@5",
        "precision@10", "recall@10", "ndcg@10", "mrr@10",
        "precision@20", "recall@20", "ndcg@20", 
        "precision@30", "recall@30", "ndcg@30", 
    ):
        summary[metric] = mean(row[metric] for row in per_query)
    return summary, per_query

In [17]:
semantic_results

[{'document': 'MINUTE N°: TRIBUNAL JUDICIAIRE DE LYON POLE SOCIAL - CONTENTIEUX GENERAL REPUBLIQUE FRANCAISE AU NOM DU PEUPLE FRANCAIS JUGEMENT DU: MAGISTRAT: ASSESSEURS: DÉBATS: PRONONCE: AFFAIRE: NUMÉRO R.G: 15 Décembre 2023 Françoise NEYMARC, présidente Dominique DALBIES, assesseur collège employeur Claude NOEL, assesseur collège salarié assistés lors des débats et du prononcé du jugement par Jean-William DUMONT, greffier tenus en audience publique le 12 Octobre 2023 jugement contradictoire, rendu en premier ressort, le 15 Décembre 2023 par le même magistrat S.A.S. [2] C/ URSSAF RHONE-ALPES N° RG 18/00237 - N° Portalis DB2H-W-B7C-S3JI DEMANDERESSE S.A.S. [2], dont le siège social est sis [Adresse 1] représentée par Me Bruno DEGUERRY, avocat au barreau de LYON, vestiaire: 1646 DÉFENDERESSE URSSAF RHONE-ALPES, dont le siège social est sis [Adresse 3] représentée par Madame [X] [M], munie d’un pouvoir Notification le: Une copie certifiée conforme à: S.A.S. [2] URSSAF RHONE-ALPES Me Bru